# Kafka Connect

## What's covered

- What Kafka Connect is and the "config not code" pitch — when to reach for it instead of writing a producer or consumer
- Source vs sink connectors
- The runtime model — workers, connectors, tasks, and how parallelism actually works
- Standalone vs distributed mode — which one to run, when
- The Connect REST API — create, inspect, pause, restart, delete connectors
- Converters — JSON, Avro, Protobuf, JSON Schema; how Schema Registry plugs in here
- Single Message Transforms (SMTs) — the in-line record edits that keep simple cases out of code
- Where source offsets live (and why sink offsets are just regular consumer offsets)
- Error handling — `errors.tolerance`, retries, dead-letter queues
- The connectors you'll meet most — Debezium for CDC, JDBC source/sink, S3 sink, Elasticsearch sink, MirrorMaker 2
- Common gotchas

## The pitch

Most Kafka pipelines you'd build by hand do the same thing: read from some external system (a database, a file, an API, a queue) and produce records, or consume records and write them somewhere else (a warehouse, an object store, a search index). The producer or consumer code is mostly boilerplate — bootstrap, batch, commit, handle retries, restart on failure — and the *actual* value is in the plumbing.

**Kafka Connect is that plumbing, factored out.** A connector is a JAR you drop into a Connect worker, configured via a JSON document, that does one of two things:

- **Source connector** — pulls data *from* an external system *into* Kafka topics. Examples: Debezium reading the Postgres write-ahead log, the JDBC source polling a table, MirrorMaker 2 mirroring another cluster.
- **Sink connector** — pulls data *from* Kafka topics *into* an external system. Examples: the S3 sink writing Parquet files, the Elasticsearch sink indexing records, the JDBC sink upserting to a table.

Connect handles the parts you'd otherwise rebuild every time: offset tracking, restart-from-failure, parallelism, schema integration with Schema Registry, configuration via REST, distributed coordination across worker nodes. **You write configuration, not code.** Most production Kafka deployments have more Connect workers than application producers and consumers combined.

When *not* to use Connect: when your data movement also requires non-trivial business logic. SMTs are good for projection, renaming, filtering — they are not good for joining two streams or calling a downstream API for enrichment. For that, write a Kafka Streams app (notebook 07) or a regular consumer.

## The runtime model — workers, connectors, tasks

Three nouns, hierarchical:

- **Worker** — a JVM process running the Connect runtime. You start workers like brokers; a *Connect cluster* is the set of workers sharing the same `group.id` (in distributed mode).
- **Connector** — a logical pipeline you create via REST. Configuration: which connector class, which source/destination, which Kafka topics, how to convert records. A connector itself does no work.
- **Task** — the unit of execution. The connector class is asked "given this config, how many tasks can you parallelize this into?" and produces 1..N task configs. Each task is assigned to one worker and runs there.

```text
         Connect cluster
  ┌──────────────────────────────────────────────────────────┐
  │                                                          │
  │  worker-1                worker-2                worker-3│
  │  ┌───────────────┐       ┌───────────────┐       ┌──────┐│
  │  │ jdbc-src-t0   │       │ jdbc-src-t1   │       │ ...  ││
  │  │ jdbc-src-t2   │       │ s3-sink-t0    │       │      ││
  │  │ debezium-t0   │       │ debezium-t1   │       │      ││
  │  └───────────────┘       └───────────────┘       └──────┘│
  └──────────────────────────────────────────────────────────┘
```

Two things to internalize:

- **Parallelism is task-count, capped by the connector's nature.** A JDBC source against five tables runs at most five tasks. A sink connector parallelizes up to the partition count of the source topic. Setting `tasks.max=10` on a connector that can only produce three task configs gets you three tasks, not ten.
- **Tasks are reassigned across workers as workers come and go.** Distributed Connect runs its own internal consumer-group-style rebalance protocol, so you can scale workers horizontally and tasks redistribute automatically.

## Standalone vs distributed mode

Connect ships in two modes, and the gap matters for production:

**Standalone mode** — one worker, one configuration file (`connect-standalone.properties` + per-connector `.properties`). Source offsets stored in a local file on disk. No cluster, no rebalance. Useful for development, demos, and edge cases where Connect is colocated with a non-distributed source like a file tail on a single host.

**Distributed mode** — N workers sharing a `group.id`. Configuration is stored in Kafka itself, in three internal topics (the names you set in worker config):

- **`config.storage.topic`** — connector configurations
- **`offset.storage.topic`** — source-connector offsets ("last LSN read," "last file position," etc.)
- **`status.storage.topic`** — task and connector status — `RUNNING`, `FAILED`, `PAUSED`

All three should be created with `replication.factor=3` and `cleanup.policy=compact` in production. They're typically prefixed with `_` (e.g. `_connect-configs`).

**For anything beyond a one-off, use distributed mode**, even with a single worker. The REST API, the in-Kafka config storage, and the ability to scale horizontally later are worth the small extra setup cost up front.

## Setup

Start a Connect worker in distributed mode alongside the existing Kafka broker. The `cp-kafka-connect` image bundles the runtime plus the standard converters; you can install additional connectors via `confluent-hub`.

```bash
docker run -d --name kafka-connect --network host \
    -e CONNECT_BOOTSTRAP_SERVERS=localhost:9092 \
    -e CONNECT_REST_PORT=8083 \
    -e CONNECT_REST_ADVERTISED_HOST_NAME=localhost \
    -e CONNECT_GROUP_ID=connect-cluster \
    -e CONNECT_CONFIG_STORAGE_TOPIC=_connect-configs \
    -e CONNECT_OFFSET_STORAGE_TOPIC=_connect-offsets \
    -e CONNECT_STATUS_STORAGE_TOPIC=_connect-status \
    -e CONNECT_CONFIG_STORAGE_REPLICATION_FACTOR=1 \
    -e CONNECT_OFFSET_STORAGE_REPLICATION_FACTOR=1 \
    -e CONNECT_STATUS_STORAGE_REPLICATION_FACTOR=1 \
    -e CONNECT_KEY_CONVERTER=org.apache.kafka.connect.json.JsonConverter \
    -e CONNECT_VALUE_CONVERTER=org.apache.kafka.connect.json.JsonConverter \
    -e CONNECT_PLUGIN_PATH=/usr/share/java,/usr/share/confluent-hub-components \
    confluentinc/cp-kafka-connect:7.7.1
```

Replication factors of 1 are for the local single-broker demo only — in production, set all three internal topics to RF=3.

Confirm the worker is up and list the installed connector plugins. The `FileStreamSourceConnector` and `FileStreamSinkConnector` ship with the worker out of the box; we'll use those for the demos.

In [ ]:
import requests, json, time

CONNECT = "http://localhost:8083"

info = requests.get(CONNECT, timeout=5).json()
print(f"version          : {info['version']}")
print(f"kafka cluster ID : {info['kafka_cluster_id']}")

# Plugin list — every connector class the worker knows how to instantiate.
plugins = requests.get(f"{CONNECT}/connector-plugins", timeout=5).json()
print("\nfirst few installed plugins:")
for p in plugins[:6]:
    print(f"  {p['type']:<14}  {p['class']}")

## The Connect REST API

Everything is HTTP. The endpoints you'll use over and over:

| Method + path | What it does |
|---|---|
| `GET /connectors` | List connector names |
| `POST /connectors` | Create a connector (body: `{name, config}`) |
| `GET /connectors/{name}` | Fetch connector config |
| `GET /connectors/{name}/status` | State of the connector + each task — `RUNNING`, `PAUSED`, `FAILED`, etc. |
| `PUT /connectors/{name}/config` | Update config (creates the connector if missing — useful for idempotent IaC) |
| `PUT /connectors/{name}/pause` | Pause without losing position |
| `PUT /connectors/{name}/resume` | Resume |
| `POST /connectors/{name}/restart?includeTasks=true` | Restart failed tasks |
| `DELETE /connectors/{name}` | Tear it down |
| `GET /connector-plugins` | What classes this worker knows |
| `PUT /connector-plugins/{class}/config/validate` | Validate a config before creating |

Below: create a `FileStreamSource` connector that watches a file and produces each new line as a record to the `connect-demo` topic.

In [ ]:
# Write a small input file the FileStreamSource can read from.
with open("/tmp/connect-input.log", "w") as f:
    f.write("line one\nline two\nline three\n")

# Create the connector — note the all-string config (Connect requires it).
config = {
    "name": "file-source-demo",
    "config": {
        "connector.class": "org.apache.kafka.connect.file.FileStreamSourceConnector",
        "tasks.max": "1",                # FileStreamSource can only run one task
        "file":      "/tmp/connect-input.log",
        "topic":     "connect-demo",
        "key.converter":   "org.apache.kafka.connect.storage.StringConverter",
        "value.converter": "org.apache.kafka.connect.storage.StringConverter",
    },
}

# Delete first if it already exists (idempotent demo).
requests.delete(f"{CONNECT}/connectors/{config['name']}")
time.sleep(1)

r = requests.post(f"{CONNECT}/connectors",
                  headers={"Content-Type": "application/json"},
                  data=json.dumps(config), timeout=5)
print("create status:", r.status_code)

In [ ]:
# Poll status until the task reports RUNNING.
for _ in range(10):
    s = requests.get(f"{CONNECT}/connectors/file-source-demo/status", timeout=5).json()
    state = s["connector"]["state"]
    tasks = [(t["id"], t["state"]) for t in s["tasks"]]
    print(f"connector={state}  tasks={tasks}")
    if state == "RUNNING" and all(t[1] == "RUNNING" for t in s["tasks"]):
        break
    time.sleep(1)

In [ ]:
# Read the records the source connector produced.
from confluent_kafka import Consumer

c = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "connect-demo-reader",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})
c.subscribe(["connect-demo"])

for _ in range(3):
    msg = c.poll(3.0)
    if msg is None: break
    if msg.error(): print("err:", msg.error()); continue
    print(f"  offset={msg.offset()}  value={msg.value().decode()}")
c.close()

## Converters — bytes ↔ structured records

Connect's internal API works on **structured records** — a typed object with a schema and a value. The broker, of course, holds **bytes**. The bridge is a **converter**, configured per connector for both keys and values:

| Converter | What it produces / consumes |
|---|---|
| `org.apache.kafka.connect.storage.StringConverter` | Plain UTF-8 strings — no schema |
| `org.apache.kafka.connect.json.JsonConverter` | JSON — optionally with an envelope `{"schema": ..., "payload": ...}` |
| `io.confluent.connect.avro.AvroConverter` | Avro + Schema Registry — the production default |
| `io.confluent.connect.protobuf.ProtobufConverter` | Protobuf + Schema Registry |
| `io.confluent.connect.json.JsonSchemaConverter` | JSON Schema + Schema Registry |

**The pairing rule:** key converter and value converter are independent. A common setup is `StringConverter` for keys (which are usually identifiers) and `AvroConverter` for values (which are typed records).

If you set `JsonConverter` with `schemas.enable=true` (the default), every record is wrapped in an envelope including the schema — verbose, but lets you decode without Schema Registry. With `schemas.enable=false`, you get plain JSON. **For production with multiple consumers, use the Avro/Protobuf/JSON-Schema converters with Schema Registry** — same rationale as notebook 05.

## Single Message Transforms (SMTs)

Connectors emit records; **SMTs** edit those records in flight before the converter touches them. Configured per connector via the `transforms` chain:

```text
  source → SMT-1 → SMT-2 → ... → converter → broker      (source connector)
  broker → converter → SMT-1 → SMT-2 → ... → sink         (sink connector)
```

A common chain on a JDBC-source connector reading a `customers` table:

```properties
transforms=route, addPrefix, dropNulls

# 1. route — rewrite topic from "customers" to "prod.customers.v1"
transforms.route.type=org.apache.kafka.connect.transforms.RegexRouter
transforms.route.regex=(.*)
transforms.route.replacement=prod.$1.v1

# 2. addPrefix — add a header for downstream tenant routing
transforms.addPrefix.type=org.apache.kafka.connect.transforms.InsertHeader
transforms.addPrefix.header=tenant
transforms.addPrefix.value.literal=acme

# 3. dropNulls — filter out tombstones from the source
transforms.dropNulls.type=org.apache.kafka.connect.transforms.Filter
transforms.dropNulls.predicate=isNull
predicates=isNull
predicates.isNull.type=org.apache.kafka.connect.transforms.predicates.RecordIsTombstone
```

**The good use cases for SMTs:** rename / drop / mask fields, route to a different topic, flatten nested records, extract a field into the message key, insert / cast / drop fields, filter tombstones.

**The wrong use cases for SMTs:** joining two streams, calling out to a service, aggregating, anything stateful. SMTs run *per record, in isolation, with no state*. Reach for Kafka Streams or a custom consumer instead.

## Source offsets and sink offsets

Two different stories on either side of Connect:

**Source connectors track "position in the source system."** That position is opaque to Kafka — it might be a Postgres LSN, a file byte offset, an HTTP cursor token, an MQ message ID. The connector reports it after producing each record; Connect stores it in `offset.storage.topic`. On restart, the connector reads its last reported position and resumes there. This is the standard at-least-once contract: a crash between produce and offset commit means the next run re-reads some records.

**Sink connectors are just consumers under the hood.** They use a normal consumer group (`connect-<connector-name>` by default) with all the offset machinery from notebook 03. The connector framework polls the consumer, hands records to the sink task, and commits offsets after the task acknowledges the batch. Same at-least-once contract; the sink had better be idempotent (most are — they upsert by primary key).

**To inspect or reset offsets:**

- Source side: read `_connect-offsets` directly, or use the offset-tracking REST endpoints (Kafka 3.6+ added `GET /connectors/{name}/offsets`, `PATCH`, `DELETE`).
- Sink side: use `kafka-consumer-groups.sh` against the consumer group, exactly as you would for any consumer.

## Error handling and dead-letter queues

Records will fail to convert, fail SMTs, or fail to write to a sink. Three knobs decide what happens:

- **`errors.tolerance`** — `none` (default) means the first failure kills the task; `all` means failures are tolerated and the connector keeps running.
- **`errors.log.enable`** + **`errors.log.include.messages`** — log the failure, optionally with the offending record.
- **`errors.deadletterqueue.topic.name`** — for sink connectors with `errors.tolerance=all`, write failed records to this topic instead of dropping them. Headers on the DLQ record describe the failure (`__connect.errors.topic`, `__connect.errors.exception.class.name`, etc.). Pair with `errors.deadletterqueue.context.headers.enable=true` to populate those headers.

**The honest production recipe:**

```properties
errors.tolerance=all
errors.log.enable=true
errors.log.include.messages=false   # records may contain PII — keep them out of logs
errors.deadletterqueue.topic.name=connect-dlq
errors.deadletterqueue.context.headers.enable=true
errors.deadletterqueue.topic.replication.factor=3
errors.retry.timeout=300000
errors.retry.delay.max.ms=60000
```

Monitor the DLQ topic — record arrival there is your signal that the source or sink hit something unexpected. A non-empty DLQ that nobody monitors is the same as silent data loss.

## The connectors you'll actually meet

A short tour of the ones that come up over and over:

**Debezium (source).** Change Data Capture from Postgres, MySQL, SQL Server, MongoDB, Oracle. Tails the database's replication log and produces one topic per table, each record being a row-level change event (`op: c|u|d`, `before`, `after`, source metadata). The default for getting OLTP state into Kafka.

**JDBC source.** Polls a SQL table on an interval — incrementing-column or timestamp mode for capturing new rows, bulk mode for full reloads. Simpler than Debezium but doesn't see deletes and doesn't capture in-between changes. Useful for append-only data and analytics-only mirroring.

**JDBC sink.** Writes Kafka records to a SQL table. Supports insert and upsert (with `pk.mode` and `insert.mode=upsert`). One of the most common sinks.

**S3 / GCS / Azure Blob sink (Confluent).** Writes batches to object storage in Parquet, Avro, or JSON. Time- or size-partitioned. The standard way to land Kafka data in a warehouse.

**Elasticsearch sink.** Indexes records into an Elasticsearch / OpenSearch index. Used for log search, analytics dashboards.

**MirrorMaker 2.** Mirrors topics from one Kafka cluster to another (DR, regional fan-out, migration). Built on Connect; you configure it like any other connector.

**HTTP sink (Confluent).** POSTs records to an arbitrary HTTP endpoint with retry and DLQ semantics. The escape hatch for systems without a dedicated connector.

## A sink example — let's pair with the source

Same pattern, reversed. The `FileStreamSinkConnector` reads from a topic and writes each record value as a line to a file. We point it at the `connect-demo` topic the source just populated.

In [ ]:
config = {
    "name": "file-sink-demo",
    "config": {
        "connector.class": "org.apache.kafka.connect.file.FileStreamSinkConnector",
        "tasks.max": "1",
        "file":   "/tmp/connect-output.log",
        "topics": "connect-demo",
        "key.converter":   "org.apache.kafka.connect.storage.StringConverter",
        "value.converter": "org.apache.kafka.connect.storage.StringConverter",
    },
}
requests.delete(f"{CONNECT}/connectors/{config['name']}")
time.sleep(1)
requests.post(f"{CONNECT}/connectors",
              headers={"Content-Type": "application/json"},
              data=json.dumps(config), timeout=5)

for _ in range(8):
    s = requests.get(f"{CONNECT}/connectors/file-sink-demo/status", timeout=5).json()
    state = s["connector"]["state"]
    tasks = [(t["id"], t["state"]) for t in s["tasks"]]
    print(f"connector={state}  tasks={tasks}")
    if state == "RUNNING" and all(t[1] == "RUNNING" for t in s["tasks"]):
        break
    time.sleep(1)

# Give the sink a couple seconds to flush, then read the output file.
time.sleep(3)
try:
    with open("/tmp/connect-output.log") as f:
        print("\nsink output:\n" + f.read())
except FileNotFoundError:
    print("sink output file not written yet — wait a bit and re-read")

In [ ]:
# Clean up the demo connectors.
for name in ("file-source-demo", "file-sink-demo"):
    r = requests.delete(f"{CONNECT}/connectors/{name}")
    print(f"deleted {name}: {r.status_code}")

## Common gotchas

- **All config values must be strings.** Even numbers and booleans. `"tasks.max": "4"`, not `4`. The REST API rejects non-string values.
- **`tasks.max` is a *maximum*, not a target.** The connector class decides the real parallelism — sometimes far less than you ask for. Watch the status output.
- **`config.storage`, `offset.storage`, `status.storage` topics with RF=1 in production.** They hold the entire Connect cluster's state. Set them to `RF=3` and `cleanup.policy=compact` before you ship.
- **Mixing JSON-with-envelope and Avro across producers and Connect consumers.** Either everyone uses Schema Registry or nobody does on a given topic. Inconsistent converters silently corrupt downstream pipelines.
- **SMT order matters, and SMTs are stateless.** They run in declared order, per record. Don't try to do anything that needs history or cross-record context.
- **Forgetting to monitor the DLQ.** A non-empty DLQ that nobody watches is silent data loss with extra steps.
- **Source offsets in distributed mode are keyed by connector name *and source partition*.** Renaming a connector resets it to the beginning. Renames are not free.
- **One Connect cluster per environment.** Sharing one Connect cluster across staging and prod is a recipe for confused tasks landing in the wrong place. Run separate clusters.

## What's next

Connect handles the "move data" half of most pipelines. The remaining half is **transform data while it's in motion** — joins, aggregations, windowed counts — which is what stream processing engines do.

- **Notebook 07 — Kafka Streams.** The library shipped with Kafka for stateful stream processing on the JVM. KStream and KTable, joins, windowing, state stores, the exactly-once stream-processing recipe.
- **Notebook 08 — Operations, Security & Performance Tuning.** Closes the curriculum: monitoring (JMX, the metrics that actually matter), SSL/SASL/ACLs, broker tuning, scaling.

Producers, consumers, topic design, schemas, Connect — that's the data-plane API surface. Streams and ops are how you make it production-grade.